# Lab 02: Cloud Deployment to GEAP Agent Runtime & Agent Registry
Building Enterprise AI Agents on **Gemini Enterprise Agent Platform (GEAP)**: Pulse Telco Workshop

---

## 🎯 Objectives
Running an agent in-process on a developer workstation is ideal for rapid prototyping, but moving to enterprise production introduces three architectural requirements: workload isolation with verifiable cryptographic identity, durable state externalized from container memory, and a centralized discovery catalog so peer agents and enterprise portals can locate the service without hardcoded URLs.

In this lab, you transition the multi-agent **Pulse Concierge** (*pulse-concierge*) from your local workstation to managed European cloud infrastructure:
1. Understand how **GEAP Agent Runtime** (*reasoningEngines*) separates serverless container execution and stateful session storage from the discovery plane in **Agent Registry**.
2. Configure *pulse-concierge/.env* to prevent environment drift between local and cloud runtimes, then deploy the multi-agent hierarchy to the workshop region (europe-west4) with its own system-attested **SPIFFE Agent Identity** (*--agent-identity*).
3. Inspect the deployed *ReasoningEngine* resource over REST, verify its SPIFFE *effectiveIdentity*, scaling bounds, and container environment variables, and grant its workload identity permission to mint delegated OAuth tokens for the two workshop user personas.
4. Invoke the cloud-hosted multi-agent endpoint across three standard surfaces (with every tool call and token count visible via *show_event*):
   * **Agents CLI** in both ADK and A2A modes
   * **Vertex AI / GEAP Python SDK** with persistent server-side **Sessions**
   * **Agent2Agent (A2A) Protocol** (Agent Card discovery and JSON-RPC message/send)
5. Explore **Agent Registry**, where your deployed agent (in europe-west4) and Google's managed **BigQuery** and **Knowledge Catalog** MCP servers (in global) are catalogued for runtime discovery.

> [!IMPORTANT]
> **First of Only Two Cloud Deployments Across the Core Workshop:**
> * **Deploy #1 (this lab)**: You deploy *pulse-concierge* v1 once right now. This provisions your managed **Agent Runtime instance** (*PULSE_AGENT_ENGINE_ID*) and its cryptographic **Agent Identity** (*PULSE_AGENT_PRINCIPAL*).
> * **Fast Local Inner Loop in Labs 03, 04, and 05 (Zero Redeploys)**: Shipping a new container image for every prompt or tool tweak kills developer velocity. Just like a production engineering team, you will develop and test v2 through v4 locally in-process while connecting over HTTPS to **this Lab 02 instance's** cloud-managed **Sessions** (*VertexAiSessionService*), **Memory Bank** (*VertexAiMemoryBankService*), and **Code Execution Sandbox** (plus remote MCP servers and Model Armor templates). That saves more than 20 minutes of waiting for Cloud Build container rebuilds.
> * **Deploy #2 (Lab 06)**: Once all MCP tools, skills, memory, guardrails, and the remote A2A NOC sub-agent are wired and verified, you perform one final cloud deployment (v5).

### Key References & Official Documentation:
* 📘 [GEAP Agent Runtime Documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime)
* 🔐 [Use Agent Identity with Agent Runtime (SPIFFE)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity) · [IAM Principal Identifiers](https://docs.cloud.google.com/iam/docs/principal-identifiers)
* 🚀 [GEAP Scale Overview (Sessions, Memory & Runtime)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale)
* 🏛️ [GEAP Govern Overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern) · [Agent Registry Documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-registry)
* 🤝 [ADK Agent2Agent (A2A) Protocol Guide](https://adk.dev/a2a/)

---

## 🏛️ Deployment, Identity & Discovery Architecture

```mermaid
graph TB
    subgraph Local["Developer Workspace"]
        Code["pulse-concierge/<br/>app/agent.py, app/tools.py & Dockerfile"]
        CLI["agents-cli deploy<br/>--agent-identity"]
    end

    subgraph GCP_EU["Google Cloud (europe-west4 & eu)"]
        Build["Cloud Build &<br/>Artifact Registry"]
        AR["GEAP Agent Runtime (europe-west4)<br/>(ReasoningEngine Instance)<br/>+ Server-Side Sessions, Memory & Sandbox"]
        ID["🔐 Agent Identity<br/>(SPIFFE Principal:<br/>principal://agents.global.org-.../reasoningEngines/ID)"]
        Model["🧠 Gemini 3.8 Flash & Live<br/>(eu Multi-Region Endpoint)"]
        REG["🗂️ Agent Registry<br/>• pulse-concierge (europe-west4)<br/>• bigquery.googleapis.com/mcp (global)<br/>• dataplex.googleapis.com/mcp (global)"]
    end

    subgraph Clients["Enterprise Clients"]
        SDK["Python SDK / CLI<br/>(ADK Stream Query)"]
        A2A["A2A Protocol Clients<br/>(/api/a2a/app)"]
        GE["Gemini Enterprise<br/>Workplace Hub (Lab 10)"]
    end

    Code --> CLI
    CLI -->|Package & Build| Build
    Build -->|Deploy Container| AR
    AR --- ID
    AR -->|Inference in EU| Model
    AR -->|Auto-Catalogued| REG
    SDK --> AR
    A2A --> AR
    REG --> GE
```

## 💶 Cost & Scaling

| Service | How it is billed (list price, verified 2026-10-20) | This lab |
| :--- | :--- | :--- |
| Agent Runtime | **\$0.085/vCPU-h + \$0.009/GiB-h** of allocated resources, rounded to the second. *"Idle time spent waiting for the next prompt between turns is not billed."* The default 4 vCPU / 4 GiB configuration costs **\$0.376 per hour of active processing**. Free each month per billing account: 50 vCPU-h and 100 GiB-h | ≈ **\$0.01** |
| Cloud Build (container build) | **\$0.006/build-minute**, **first 2,500 build-minutes per billing account per month free** | ≈ **\$0.00** (~5 min) |
| Artifact Registry (image) | **\$0.10/GiB-month**, first 0.5 GiB free per billing account. Same-region transfer is free | < **\$0.01** for the workshop days |
| Cloud Storage staging bucket (europe-west4) | **\$0.020/GB-month** Standard (no Always Free tier in Europe) | ≈ **\$0.00** |
| Sessions | Storage **\$0.30/GiB-month**. Reads 1 vCPU-h (\$0.085) per 3M; writes 1 vCPU-h per 1M | ≈ **\$0.00** |
| Agent Registry, A2A card | Agent Registry: **no public price**. A2A on Agent Runtime is billed as Agent Runtime | \$0.00 |
| Gemini 3.8 Flash (eu) | \$1.65 / \$8.25 per 1M input / output tokens (50% introductory credit through 2026-12-31) | ~5 turns ≈ **\$0.30** |

**How it scales:**
* Minimum instances range from 0 to 10 (default 1). In Preview, idle minimum instances are **not billed**; this policy is subject to change at GA.
* Maximum instances range from 1 to 1000 (default 100). CPU options are 1, 2, 4, 6, or 8 vCPU and memory ranges from 1 to 32 GiB (default 4 vCPU / 4 GiB). Default container concurrency is 9 (*2 x cpu + 1*).
* Interactive *streamQuery* calls run up to **10 minutes**. Long-running asynchronous jobs run up to 7 days.
* Regional quotas per project: **90 queries/min**, **10 deploys/min**, 100 agent resources, and A2A 60 POST / 600 GET per minute. Quotas are shared when multiple participants work in the same project.

**What this lab cost you: ≈ \$0.30 to \$0.40.**

*Pricing references:* [Gemini Enterprise Agent Platform pricing](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing) · [Cloud Build pricing](https://cloud.google.com/build/pricing) · [Artifact Registry pricing](https://cloud.google.com/artifact-registry/pricing) · [Gemini platform generative AI pricing](https://cloud.google.com/gemini-enterprise-agent-platform/generative-ai/pricing)

## Step 2.0: Load Workshop Configuration & Event Printer

We load the shared environment variables from *../workshop.env*, configure the regional infrastructure endpoint in *europe-west4* alongside the European Union multi-region model endpoints (*eu*), and define the *show_event* and *show_stats* helpers so every streamed tool call, tool response, and token count is visible.

In [ ]:
# Step 2.0: Load workshop configuration and define show_event() / show_stats() helpers
import os, sys, json, pathlib, warnings
os.environ["PATH"] = f"{pathlib.Path.home() / '.local/bin'}:{os.environ.get('PATH', '')}"
os.environ.pop("VIRTUAL_ENV", None)  # prevent parent .venv from leaking into child uv / agents-cli calls
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)  # vertexai.Client is the legacy alias of agentplatform.Client

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
ENV_FILE = REPO_ROOT / "workshop.env"
assert ENV_FILE.exists(), "workshop.env not found - run Lab00 first"
load_dotenv(ENV_FILE, override=True)

PROJECT_ID            = os.environ["PROJECT_ID"]
PROJECT_NUMBER        = os.environ["PROJECT_NUMBER"]
REGION                = os.environ.get("REGION", "europe-west4")       # europe-west4: Agent Runtime, Sessions, Memory Bank, Sandbox, Registry
MODEL_LOCATION        = os.environ.get("MODEL_LOCATION", "eu")         # eu: multi-region endpoint for gemini-3.8-flash
LIVE_LOCATION         = os.environ.get("LIVE_LOCATION", "eu")          # eu: multi-region endpoint for gemini-3.8-live
BQ_LOCATION           = os.environ.get("BQ_LOCATION", "EU")            # EU: BigQuery multi-region
MODEL                 = os.environ.get("MODEL", "gemini-3.8-flash")
AGENT_NAME            = os.environ.get("AGENT_NAME", "pulse-concierge")
BQ_DATASET_CRM        = os.environ.get("BQ_DATASET_CRM", "telco_crm")
BQ_DATASET_BILLING    = os.environ.get("BQ_DATASET_BILLING", "telco_billing")
BQ_DATASET_NETWORK    = os.environ.get("BQ_DATASET_NETWORK", "telco_network")
PERSONA_ADMIN_SA      = os.environ.get("PERSONA_ADMIN_SA", f"persona-alex-exec@{PROJECT_ID}.iam.gserviceaccount.com")
PERSONA_SPECIALIST_SA = os.environ.get("PERSONA_SPECIALIST_SA", f"persona-elena-care@{PROJECT_ID}.iam.gserviceaccount.com")
AGENT_DIR             = REPO_ROOT / AGENT_NAME

# Why GOOGLE_API_USE_CLIENT_CERTIFICATE=false: on client-cert-enrolled machines, google-genai / vertexai otherwise
# attempts *.rep.mtls.googleapis.com, which fails DNS or crashes the aiohttp connector (harmless outside corp machines).
os.environ.setdefault("GOOGLE_API_USE_CLIENT_CERTIFICATE", "false")
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "LIVE_LOCATION": LIVE_LOCATION,
    "BQ_LOCATION": BQ_LOCATION,
    "PERSONA_ADMIN_SA": PERSONA_ADMIN_SA,
    "PERSONA_SPECIALIST_SA": PERSONA_SPECIALIST_SA,
    "GOOGLE_GENAI_USE_VERTEXAI": "true",
    "GOOGLE_API_USE_CLIENT_CERTIFICATE": "false",
    "UV_DEFAULT_INDEX": "https://pypi.org/simple",
    "CLOUDSDK_CORE_PROJECT": PROJECT_ID,
    "CLOUDSDK_CORE_DISABLE_PROMPTS": "1",
})
print(f"Project: {PROJECT_ID} ({PROJECT_NUMBER}) | region: {REGION} | model: {MODEL} @ {MODEL_LOCATION} | live @ {LIVE_LOCATION}")
print(f"Agent dir: {AGENT_DIR} | BQ datasets ({BQ_LOCATION}): {BQ_DATASET_CRM}, {BQ_DATASET_BILLING}, {BQ_DATASET_NETWORK}")

def terminal(cmd, cwd=None):
    """Print the exact command a cell is about to run, so you can copy/paste it into your own terminal."""
    prefix = f"cd {os.path.relpath(cwd, REPO_ROOT)} && " if cwd else ""
    print(f"$ {prefix}{cmd}\n")


import json


def show_event(ev, stats=None, text_limit=2000):
    """Print one agent event: tool calls, tool results, errors and text.

    Works for local ADK `Event` objects (Runner.run_async / run_live) and for the
    dicts a deployed agent streams (async_stream_query), which use snake_case keys.
    Pass a dict as `stats` to accumulate token usage and the number of model calls.
    """
    d = ev if isinstance(ev, dict) else ev.model_dump(exclude_none=True)
    who = d.get("author", "agent")
    err_code = d.get("error_code") or d.get("errorCode")
    err_msg = d.get("error_message") or d.get("errorMessage")
    if err_code or err_msg:
        print(f"  !! [{who}] ERROR {err_code or ''}: {str(err_msg or '')[:400]}")
    for part in (d.get("content") or {}).get("parts") or []:
        fc = part.get("function_call") or part.get("functionCall")
        fr = part.get("function_response") or part.get("functionResponse")
        if fc:
            args = json.dumps(fc.get("args") or {}, ensure_ascii=False, default=str)
            print(f"  -> [{who}] tool call   {fc.get('name')}({args[:300]})")
        elif fr:
            resp = json.dumps(fr.get("response") or {}, ensure_ascii=False, default=str)
            print(f"  <- [{who}] tool result {fr.get('name')}: {resp[:300]}")
        elif part.get("text") and not part.get("thought"):
            print(f"  [{who}] {part['text'].strip()[:text_limit]}")
    usage = d.get("usage_metadata") or d.get("usageMetadata")
    if stats is not None and usage:
        stats["model_calls"] = stats.get("model_calls", 0) + 1
        for key in ("prompt_token_count", "candidates_token_count", "thoughts_token_count"):
            stats[key] = stats.get(key, 0) + (usage.get(key) or usage.get(key.replace("_token_count", "TokenCount")) or 0)


def show_stats(stats):
    if stats:
        print(f"  (model calls: {stats.get('model_calls', 0)} | tokens in: {stats.get('prompt_token_count', 0):,} "
              f"| out: {stats.get('candidates_token_count', 0):,} | thinking: {stats.get('thoughts_token_count', 0):,})")

## Step 2.1: Understanding GEAP Agent Runtime & SPIFFE Agent Identity

### Why Workload Identity and Managed Runtimes Matter
In production, you must solve two structural problems: how to scale stateless agent containers while keeping multi-turn session state durable, and how to give each agent its own least-privilege cryptographic identity.

A **common anti-pattern** in early cloud deployments is running every agent container under a single shared IAM service account (or embedding static API keys in environment variables). When ten agents share one service account, a prompt injection or misconfiguration in any single agent inherits the permissions of all ten, and Cloud Audit Logs cannot distinguish which specific agent instance accessed a dataset.

### Mechanics of Agent Runtime
[**Agent Runtime**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime) is the managed, serverless execution environment of Gemini Enterprise Agent Platform. The same underlying Google Cloud resource appears under three names depending on the surface: the product name is **Agent Runtime** (formerly Agent Engine), the REST API resource collection is *reasoningEngines*, and the Python SDK accessor is *client.agent_engines*.

* **Container-Native Serverless Hosting**: You provide your ADK project directory and its Dockerfile. Agent Runtime packages the source tree, builds the container image in Cloud Build, stores it in Artifact Registry, deploys the container in *europe-west4*, manages horizontal autoscaling, and fronts the workload with an authenticated HTTPS proxy.
* **Scaling, Concurrency & Request Timeouts**:
  * Instance bounds are controlled by *min_instances* (0 to 10, default 1; note that *agents-cli deploy* defaults to 0 unless *--min-instances* is passed) and *max_instances* (1 to 1000, default 100).
  * Each container instance defaults to **4 vCPU and 4 GiB RAM** with a concurrency of 9 requests per container (2 x cpu + 1).
  * When minimum instances is 0, the runtime scales to zero when idle, trading zero idle compute cost for a multi-second cold start on the first request after a quiet period. Passing *--min-instances 1* keeps a warm instance ready (and during Preview, idle minimum instances are not billed).
  * Interactive streaming and bidirectional voice calls have a maximum duration of **10 minutes** (long-running background asynchronous jobs can run for up to 7 days).
* **Built-In Stateful Context Subsystems (Reused in Labs 03 to 06)**: Every Agent Runtime instance hosts managed **Sessions** (multi-turn conversation event logs and the session state dictionary), **Memory Bank** (long-term cross-session user memories), and **Code Execution Sandboxes** (isolated execution environments explored in Lab 04).
* **[SPIFFE-Based Agent Identity (*--agent-identity*)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity)**: Passing *--agent-identity* during deployment provisions a dedicated, system-attested workload identity backed by **SPIFFE**, with Google automatically rotating its X.509 certificate:
  ```text
  principal://agents.global.org-<ORG_ID>.system.id.goog/resources/aiplatform/projects/<PROJECT_NUMBER>/locations/<REGION>/reasoningEngines/<ENGINE_ID>
  ```
  The trust domain is organization-scoped when the project belongs to a Google Cloud Organization, or project-scoped otherwise. You never construct this string by hand; we read the exact URI from *spec.effectiveIdentity* on the deployed resource.

## Step 2.2: Prepare the Agent Environment Configuration for Cloud Deployment

When *agents-cli deploy* packages *pulse-concierge/*, it reads *pulse-concierge/.env* and injects those key-value pairs as container environment variables on the cloud runtime. However, Agent Runtime intercepts reserved platform keys such as *GOOGLE_CLOUD_PROJECT* and overwrites them inside the container with the numeric project number rather than the alphanumeric project ID string.

If your tool code builds service account emails dynamically from *GOOGLE_CLOUD_PROJECT*, that silent substitution breaks IAM impersonation once deployed to the cloud. We therefore ensure four explicit configuration groups are written to *pulse-concierge/.env* before building the container:
1. **Explicit Persona Service Account Emails (PERSONA_ADMIN_SA & PERSONA_SPECIALIST_SA)**: Because *GOOGLE_CLOUD_PROJECT* inside the cloud container resolves to the numeric project number, *app/tools.py* reads the full alphanumeric service account emails directly from these two environment variables.
2. **EU Data Residency Pinning (LIVE_LOCATION=eu & BQ_LOCATION=EU)**: Keeps Gemini 3.8 Live voice sessions and BigQuery SQL jobs pinned to the European Union multi-region.
3. **[mTLS Opt-Out for EU Regional Endpoints (GOOGLE_API_USE_CLIENT_CERTIFICATE=false & GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES=false)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity)**: Agent Identity tokens are certificate-bound by default for Context-Aware Access. Because the *eu* multi-region endpoint (*aiplatform.eu.rep.googleapis.com*) does not expose a client-certificate mTLS host, setting these two flags allows the cloud container to call the *eu* endpoint cleanly.
4. **Hermetic Container Startup (UV_NO_SYNC=1)**: Instructs *uv* inside the container to run directly against the pre-built virtual environment layer without attempting an outbound PyPI check during cold start.

In [ ]:
# Step 2.2: Prepare pulse-concierge/.env for cloud deployment (checking each required key independently)
env_path = AGENT_DIR / ".env"
existing_lines = env_path.read_text().splitlines() if env_path.exists() else []
env_map = {}
for line in existing_lines:
    if "=" in line and not line.strip().startswith("#"):
        k, v = line.split("=", 1)
        env_map[k.strip()] = v.strip()

env_map.update({
    "GOOGLE_GENAI_USE_VERTEXAI": "1",
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": MODEL_LOCATION,
    "LIVE_LOCATION": LIVE_LOCATION,
    "BQ_LOCATION": BQ_LOCATION,
    "GOOGLE_API_USE_CLIENT_CERTIFICATE": "false",
    "GOOGLE_API_PREVENT_AGENT_TOKEN_SHARING_FOR_GCP_SERVICES": "false",
    "UV_NO_SYNC": "1",
    "PERSONA_ADMIN_SA": PERSONA_ADMIN_SA,
    "PERSONA_SPECIALIST_SA": PERSONA_SPECIALIST_SA,
})

env_path.write_text("\n".join(f"{k}={v}" for k, v in env_map.items()) + "\n")
print("pulse-concierge/.env ready for deployment:")
print(env_path.read_text())

### Step 2.2b: Deploy Pulse Concierge v1 to Agent Runtime (5 to 8 minutes)

Now we deploy *pulse-concierge* to **Agent Runtime** in the workshop region (*europe-west4*) with *--agent-identity*. On the wire, *agents-cli deploy* archives the source directory, submits a Cloud Build job to build the container image in Artifact Registry, creates (or updates) the *ReasoningEngine* resource, and writes the resulting identifiers to *pulse-concierge/deployment_metadata.json*.

> [!NOTE]
> **Idempotency & FORCE_REDEPLOY**:
> If *deployment_metadata.json* already points to an active *ReasoningEngine* in your project, the cell below skips re-running the 6-minute Cloud Build unless you set *FORCE_REDEPLOY = True*. Whenever you modify *app/agent.py*, *app/tools.py*, or *pulse-concierge/.env* and want to push a new container revision to the cloud, set *FORCE_REDEPLOY = True*.

To run this deployment from your terminal:
```bash
cd pulse-concierge
agents-cli deploy \
  --project "$PROJECT_ID" \
  --region "$REGION" \
  --agent-identity \
  --no-confirm-project
```

In [ ]:
# Step 2.2b: Deploy the agent to Agent Runtime with its own SPIFFE Agent Identity (takes 5 to 8 minutes)
import subprocess, json, time, requests, google.auth
from google.auth.transport.requests import Request

meta_file = AGENT_DIR / "deployment_metadata.json"
if meta_file.exists():
    meta_txt = meta_file.read_text()
    if f"projects/{PROJECT_NUMBER}/" not in meta_txt and f"projects/{PROJECT_ID}/" not in meta_txt:
        print(f"Removing stale deployment_metadata.json from previous project before deploying to {PROJECT_ID}...")
        meta_file.unlink()

FORCE_REDEPLOY = False   # set True after changing agent code or .env to push a new container revision
already_active = False
if meta_file.exists() and "pending_operation" in json.loads(meta_file.read_text()):
    print("Resolving in-progress deployment operation via agents-cli deploy --status...")
    while "pending_operation" in json.loads(meta_file.read_text()):
        sr = subprocess.run("agents-cli deploy --status", shell=True, cwd=AGENT_DIR, env={**os.environ, "VIRTUAL_ENV": ""}, text=True, capture_output=True)
        print((sr.stdout or sr.stderr).strip()[-1000:])
        if "pending_operation" in json.loads(meta_file.read_text()):
            time.sleep(15)
if meta_file.exists():
    rid = json.loads(meta_file.read_text()).get("remote_agent_runtime_id", "")
    if rid:
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
        creds.refresh(Request())
        resp = requests.get(f"https://{REGION}-aiplatform.googleapis.com/v1/{rid}", headers={"Authorization": f"Bearer {creds.token}"})
        if resp.status_code == 200 and resp.json().get("spec", {}).get("effectiveIdentity") and resp.json().get("spec", {}).get("sourceCodeSpec"):
            print(f"Agent Runtime already deployed in {PROJECT_ID}: {rid}")
            already_active = True

if already_active and not FORCE_REDEPLOY:
    print("Skipping deploy (set FORCE_REDEPLOY = True after changing code/.env to redeploy).")
else:
    cmd = f"agents-cli deploy --project {PROJECT_ID} --region {REGION} --agent-identity --no-confirm-project"
    terminal(cmd, cwd=AGENT_DIR)
    r = subprocess.run(cmd, shell=True, cwd=AGENT_DIR, env={**os.environ, "VIRTUAL_ENV": ""}, text=True, capture_output=True)
    print(r.stdout[-3500:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError("deploy failed")

### Step 2.2c: Persist the Deployed Agent Runtime Identifiers to workshop.env

When the deployment completes, *agents-cli deploy* writes the canonical resource name to *pulse-concierge/deployment_metadata.json*. We extract *PULSE_AGENT_ENGINE*, *PULSE_AGENT_ENGINE_ID*, and *PULSE_AGENT_URL* and save them to *workshop.env* so Labs 03 through 08 can target this exact cloud instance.

In [ ]:
# Step 2.2c: Record the deployed instance identifiers in workshop.env for subsequent labs
def save_to_workshop_env(**kv):
    """Persist values for the next labs (workshop.env) and for this kernel."""
    lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
    for k, v in kv.items():
        lines = [l for l in lines if not l.startswith(f"{k}=")]
        lines.append(f"{k}={v}")
        os.environ[k] = str(v)
    ENV_FILE.write_text("\n".join(lines) + "\n")
    print("saved:", ", ".join(f"{k}={v}" for k, v in kv.items()))

meta = json.loads((AGENT_DIR / "deployment_metadata.json").read_text())
PULSE_AGENT_ENGINE    = meta["remote_agent_runtime_id"]          # projects/NUM/locations/REGION/reasoningEngines/ID
PULSE_AGENT_ENGINE_ID = PULSE_AGENT_ENGINE.split("/")[-1]
PULSE_AGENT_URL       = f"https://{REGION}-aiplatform.googleapis.com/v1/{PULSE_AGENT_ENGINE}"

save_to_workshop_env(PULSE_AGENT_ENGINE=PULSE_AGENT_ENGINE, PULSE_AGENT_ENGINE_ID=PULSE_AGENT_ENGINE_ID, PULSE_AGENT_URL=PULSE_AGENT_URL)

print()
print("Console (deployments):", f"https://console.cloud.google.com/agent-platform/runtimes?project={PROJECT_ID}")
print("Console (this agent): ", f"https://console.cloud.google.com/vertex-ai/agents/agent-engines/locations/{REGION}/agent-engines/{PULSE_AGENT_ENGINE_ID}?project={PROJECT_ID}")

### Step 2.2d: Inspect the Deployed ReasoningEngine Resource & Grant IAM Roles to Its SPIFFE Agent Identity

The cell below queries the Vertex AI REST API to inspect what Agent Runtime provisioned on the control plane:
* **spec.identityType**: Set to "AGENT_IDENTITY".
* **spec.effectiveIdentity**: The system-attested SPIFFE principal (*principal://agents.global.org-...*) dedicated to this specific agent instance.
* **spec.deploymentSpec**: The container environment variables (confirming the persona emails, EU location settings, and mTLS flags are present) along with CPU, memory, and instance scaling parameters.

Why do we grant *roles/iam.serviceAccountTokenCreator* on the two persona service accounts to *PULSE_AGENT_PRINCIPAL* below?
In Lab 01, your local notebook process ran under your personal developer Application Default Credentials (ADC), which called the IAM Credentials API to mint short-lived OAuth tokens for *persona-alex-exec* and *persona-elena-care*. Now that the agent runs inside **Agent Runtime**, the caller requesting those delegated tokens is the container's own workload identity (*PULSE_AGENT_PRINCIPAL*). Granting Token Creator permission to *PULSE_AGENT_PRINCIPAL* authorizes the deployed container to mint short-lived persona tokens on behalf of signed-in users (Alex and Elena) while BigQuery IAM enforces each persona's dataset permissions at the storage layer.

Equivalent terminal command:
```bash
curl -s "https://${REGION}-aiplatform.googleapis.com/v1/$PULSE_AGENT_ENGINE" \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" | jq '.spec | {identityType, effectiveIdentity}'
```

In [ ]:
# Step 2.2d: Inspect the deployed instance through the REST API and grant Agent Identity permissions
import requests, google.auth
from google.auth.transport.requests import Request

def gcp_token():
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    creds.refresh(Request())
    return creds.token

engine = requests.get(PULSE_AGENT_URL, headers={"Authorization": f"Bearer {gcp_token()}"}).json()
spec = engine.get("spec", {})
identity = spec.get("effectiveIdentity", "")
PULSE_AGENT_PRINCIPAL = identity if identity.startswith("principal://") else "principal://" + identity
save_to_workshop_env(PULSE_AGENT_PRINCIPAL=PULSE_AGENT_PRINCIPAL)

print("display name :", engine.get("displayName"))
print("update time  :", engine.get("updateTime"))
print("identity     :", PULSE_AGENT_PRINCIPAL, "  <- the agent's own SPIFFE principal (Agent Identity)")
print("env vars     :", {e["name"]: e.get("value") for e in spec.get("deploymentSpec", {}).get("env", [])})
print("resources    :", spec.get("deploymentSpec", {}).get("resourceLimits"), "| scaling:", {k: v for k, v in spec.get("deploymentSpec", {}).items() if "nstance" in k})

# Grant the deployed Agent Identity permission to mint delegated OAuth 2.0 tokens for the 2 workshop user personas
# and run BigQuery jobs (also used in Lab 03+ when querying via BigQuery MCP):
for role in ["roles/bigquery.jobUser", "roles/bigquery.dataViewer"]:
    subprocess.run(
        f'gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{PULSE_AGENT_PRINCIPAL}" --role={role} --condition=None --quiet >/dev/null',
        shell=True, check=True,
    )
for persona_sa in [PERSONA_ADMIN_SA, PERSONA_SPECIALIST_SA]:
    subprocess.run(
        f'gcloud iam service-accounts add-iam-policy-binding {persona_sa} --member="{PULSE_AGENT_PRINCIPAL}" --role=roles/iam.serviceAccountTokenCreator --project={PROJECT_ID} --quiet >/dev/null',
        shell=True, check=True,
    )
print("Granted Agent Identity TokenCreator on persona-alex-exec & persona-elena-care.")

## Step 2.3: Invoke the Deployed Multi-Agent System Across Three Surfaces

Once deployed on Agent Runtime, *pulse-concierge* exposes three standard invocation surfaces from the same container deployment:
1. **CLI (*agents-cli run --url ...*)**: For rapid terminal smoke tests in both native ADK mode (*--mode adk*) and A2A mode (*--mode a2a*).
2. **Vertex AI / GEAP Python SDK (*client.agent_engines*)**: For application backends requiring managed server-side **Sessions** and authenticated *user_id* propagation ("alex" and "elena").
3. **Agent2Agent (A2A) Protocol (*/api/a2a/app*)**: For cross-agent interoperability and Gemini Enterprise workplace registration.

### 2.3a) From the Terminal: Agents CLI in ADK and A2A Modes

Just as in Lab 01, *agents-cli run* sends a fixed caller identifier of *user_id = "cli-user"* (and unauthenticated A2A requests map to a synthetic context user ID). Consequently, *agents-cli run --url* is ideal for verifying **Knowledge Catalog** lookups over both ADK and A2A modes, whereas per-user BigQuery queries (as "alex" or "elena") are executed via the Python SDK in Step 2.3b below.

```bash
cd pulse-concierge
agents-cli run --url "$PULSE_AGENT_URL" --mode adk \
  "Using Knowledge Catalog, what are the official Pulse Telco definitions for High Churn Risk Subscriber and Delinquent Account Exposure?"
agents-cli run --url "$PULSE_AGENT_URL" --mode a2a \
  "Using Knowledge Catalog, which BigQuery tables store subscriber churn risk scores and monthly invoices?"
```

In [ ]:
# Step 2.3a: Invoke the deployed agent from the CLI in both --mode adk and --mode a2a
def remote_run(prompt, mode="adk", extra=""):
    cmd = f'agents-cli run --url {PULSE_AGENT_URL} --mode {mode} {extra} "{prompt}"'.replace("  ", " ")
    terminal(cmd, cwd=AGENT_DIR)
    r = subprocess.run(cmd, shell=True, cwd=AGENT_DIR, env={**os.environ, "VIRTUAL_ENV": ""}, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())

print("=== CLI Mode 1: --mode adk ===")
remote_run("Using Knowledge Catalog, what are the official Pulse Telco definitions for High Churn Risk Subscriber and Delinquent Account Exposure?", mode="adk")

print("\n=== CLI Mode 2: --mode a2a ===")
remote_run("Using Knowledge Catalog, which BigQuery tables store subscriber churn risk scores and monthly invoices?", mode="a2a")

### 2.3b) From Python: Persistent Server-Side Sessions & Per-Persona BigQuery IAM on Agent Runtime

In Lab 01, *InMemorySessionService* kept conversation history in local process RAM; restarting the notebook kernel or scaling across multiple container replicas would lose all prior turns. On **Agent Runtime**, sessions are persisted server-side in *europe-west4* as managed Google Cloud resources via *VertexAiSessionService*:
* **Turn 1 (user_id="alex", shared session)**: Alex asks for Prague (PRG) High Churn Risk subscribers and Active Subscription ARPU (HTTP 200 OK as *persona-alex-exec*).
* **Turn 2 (user_id="alex", same session)**: Alex asks for 2026-08 Delinquent versus Disputed billing exposure (HTTP 200 OK as *persona-alex-exec*).
* **Turn 3 (user_id="alex", same session)**: Alex asks a follow-up question that requires synthesizing the server-side session history accumulated across Turns 1 and 2.
* **Turn 4 (user_id="elena", separate session)**: Elena asks for 2026-08 overdue invoices in *telco_billing*. The cloud container impersonates *persona-elena-care*, and BigQuery IAM blocks the query with **HTTP 403 Access Denied**.

Observe how *show_event(event, stats)* prints every streamed sub-agent delegation, tool call, tool response, final answer, and token count.

In [ ]:
# Step 2.3b: Invoke the deployed agent from Python with server-side Sessions and show_event()
import asyncio, vertexai

client = vertexai.Client(project=PROJECT_ID, location=REGION)
remote_agent = client.agent_engines.get(name=PULSE_AGENT_ENGINE)

async def ask(user_id: str, session_id: str, text: str):
    print(f"\n{'=' * 88}\nUSER  (user_id={user_id!r}, session={session_id}):\n  {text}\n{'-' * 88}")
    stats = {}
    async for event in remote_agent.async_stream_query(user_id=user_id, session_id=session_id, message=text):
        show_event(event, stats)
    show_stats(stats)

session_alex = await remote_agent.async_create_session(user_id="alex")
SESSION_ID = session_alex["id"]
print("Created server-side session for alex:", SESSION_ID)

# Turn 1 (alex -> telco_crm allowed)
await ask("alex", SESSION_ID, "Check how many High Churn Risk subscribers we have in Prague (PRG) and their Active Subscription ARPU.")

# Turn 2 (alex -> telco_billing allowed)
await ask("alex", SESSION_ID, "Now check our 2026-08 billing invoices: what is the Delinquent (>=30d overdue) vs Disputed exposure?")

# Turn 3 (alex -> server-side session memory across Turns 1 & 2)
await ask("alex", SESSION_ID, "Which region had the High Churn Risk subscribers we checked in my first question, and how does its MRR at risk compare to our 2026-08 Delinquent exposure?")

# Turn 4 (elena -> telco_billing denied by BigQuery IAM HTTP 403 in the cloud)
session_elena = await remote_agent.async_create_session(user_id="elena")
await ask("elena", session_elena["id"], "Show me the 2026-08 overdue invoices in telco_billing.")

### 2.3c) Inspect Server-Side Session Resources (async_list_sessions & async_get_session)

Because sessions live on the *ReasoningEngine* resource in *europe-west4* outside container memory, you can list active sessions for any user ID and inspect the persisted event log (event count, timestamps, tool calls, and session state). In Labs 03 through 06, our local *Runner* will read and write to this exact same server-side session store.

In [ ]:
# Step 2.3c: List server-side sessions and inspect stored session events
sessions = await remote_agent.async_list_sessions(user_id="alex")
for s in sessions["sessions"]:
    print("session", s["id"], "| user", s["userId"], "| last update", time.strftime("%H:%M:%S", time.gmtime(s["lastUpdateTime"])))

detail = await remote_agent.async_get_session(user_id="alex", session_id=SESSION_ID)
print(f"events stored server-side in session {SESSION_ID}: {len(detail['events'])}")

### 2.3d) Over the Agent2Agent (A2A) Protocol (/api/a2a/app)

When different teams build specialist agents across separate repositories or runtimes, coupling them through internal Python imports is impossible. Every ADK project scaffolded by *agents-cli* automatically mounts the open **[Agent2Agent (A2A)](https://adk.dev/a2a/)** protocol adapter (*app/app_utils/a2a.py*). Agent Runtime proxies HTTP paths of your container under */reasoningEngines/v1/{name}/api/...*:
* **GET .../api/a2a/app/.well-known/agent-card.json**: Returns the machine-readable **Agent Card** advertising the agent's name, description, version, capabilities, and skills (ADK auto-generates one A2A skill entry per tool or sub-agent, where "model" represents the root orchestrator LLM itself).
* **POST .../api/a2a/app**: Accepts standard JSON-RPC 2.0 *message/send* requests from peer agents or enterprise orchestrators.

Equivalent curl command to fetch the Agent Card from your terminal:
```bash
curl -s "https://${REGION}-aiplatform.googleapis.com/reasoningEngines/v1/$PULSE_AGENT_ENGINE/api/a2a/app/.well-known/agent-card.json" \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" | jq .
```

In [ ]:
# Step 2.3d: Query the deployed agent over the A2A protocol (Agent Card and JSON-RPC message/send)
A2A_BASE = f"https://{REGION}-aiplatform.googleapis.com/reasoningEngines/v1/{PULSE_AGENT_ENGINE}/api/a2a/app"
headers = {"Authorization": f"Bearer {gcp_token()}", "Content-Type": "application/json"}

card = requests.get(f"{A2A_BASE}/.well-known/agent-card.json", headers=headers).json()
print("agent card:", json.dumps({k: card[k] for k in ("name", "description", "version", "capabilities")}, indent=2))
print("skills    :", [s["name"] for s in card.get("skills", [])])

rpc = {"jsonrpc": "2.0", "id": "1", "method": "message/send",
       "params": {"message": {"role": "user", "messageId": "m1",
                              "parts": [{"kind": "text", "text": "Using Knowledge Catalog, how does Pulse Telco define a High Churn Risk Subscriber? One sentence."}]}}}
task = requests.post(A2A_BASE, headers=headers, json=rpc).json()["result"]
print("\nA2A task state:", task["status"]["state"])
for artifact in task.get("artifacts", []):
    for part in artifact.get("parts", []):
        if part.get("kind") == "text":
            print("Pulse (via A2A):", part["text"])

## Step 2.4: Explore the Regional & Global Agent Registry

### 🏛️ Why Centralized Discovery Matters at Enterprise Scale
A single prototype agent with hardcoded tool endpoints is easy to track in one repository. As an enterprise scales to dozens of domain agents and shared MCP servers across business units, hardcoded URLs create brittle point-to-point dependencies and unmanaged shadow IT. 

**[Agent Registry](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-registry)** (*agentregistry.googleapis.com*) serves as the centralized system of record and discovery catalog (analogous in role to an enterprise API catalog for agents and MCP servers):
1. **Automatic Discovery of Agent Runtime Deployments**: Deploying *pulse-concierge* to *europe-west4* automatically registers it in the regional Agent Registry (*locations/europe-west4/agents*). You may also see a default system entry named Workspace Agent alongside *pulse-concierge*, which Google Cloud provisions automatically in every project.
2. **Pre-Catalogued Google-Managed MCP Servers**: Google's managed remote MCP endpoints, including **BigQuery MCP** (*bigquery.googleapis.com/mcp*) and **Knowledge Catalog MCP** (*dataplex.googleapis.com/mcp*), are pre-registered under *locations/global/mcpServers*.
3. **Dynamic Runtime Resolution**: In Lab 03 and Lab 06, our agents resolve MCP servers, Skills, and A2A peers directly from Agent Registry by logical identifier rather than hardcoding infrastructure URLs.

Equivalent terminal commands:
```bash
gcloud agent-registry agents list --project="$PROJECT_ID" --location="$REGION"
gcloud agent-registry mcp-servers list --project="$PROJECT_ID" --location=global
```

In [ ]:
# Step 2.4: List the regional agent entry and global Google-managed MCP servers in Agent Registry
cmd = f"gcloud agent-registry agents list --project={PROJECT_ID} --location={REGION} --format='table(name.basename(),displayName,createTime)'"
terminal(cmd); print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

cmd = f"gcloud agent-registry mcp-servers list --project={PROJECT_ID} --location=global --format='value(displayName)' | grep -E 'bigquery|dataplex' || true"
terminal(cmd); print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

print("Console:", f"https://console.cloud.google.com/agent-platform/agent-registry?project={PROJECT_ID}")

## 🧪 Try It Yourself (5 minutes or less each)

1. **Exercise 1: Prove the session is persisted server-side across kernel restarts**:
   * Copy the session ID printed in Step 2.3b above.
   * Restart your Jupyter kernel (**Kernel -> Restart Kernel**), run **only** Step 2.0 (cell-01) and Step 2.2c (cell-06), and then run:
     ```python
     import vertexai
     client = vertexai.Client(project=PROJECT_ID, location=REGION)
     remote_agent = client.agent_engines.get(name=PULSE_AGENT_ENGINE)
     async for ev in remote_agent.async_stream_query(user_id="alex", session_id="<PASTE_SESSION_ID>", message="What region and billing period did we just analyze?"):
         show_event(ev)
     ```
   * Even after your local Python process memory was wiped, *VertexAiSessionService* in *europe-west4* retains the full conversation history.
2. **Exercise 2: Verify the Agent Runtime's SPIFFE identity in Cloud Audit Logs**:
   * In Lab 01, *serviceAccountDelegationInfo* showed your personal developer ADC account as the impersonator of *persona-alex-exec*. Run an audit log query after Step 2.3b:
     ```bash
     gcloud logging read 'logName:"cloudaudit.googleapis.com%2Fdata_access" AND protoPayload.serviceName="bigquery.googleapis.com" AND protoPayload.authenticationInfo.principalEmail:"persona-alex-exec@"' \
       --project="$PROJECT_ID" --limit=3 --format="yaml(timestamp,protoPayload.authenticationInfo)"
     ```
   * Observe that the delegated token was minted by the deployed Agent Runtime's SPIFFE identity (*principal://agents.global.org-...*).
3. **Exercise 3: Inspect scaling flags (--min-instances and --max-instances)**:
   * Run the following command in a scratch cell to inspect the deployment scaling flags:
     ```python
     !agents-cli deploy --help
     ```
   * Keep minimum instances at 0 (or 1 in Preview) during the workshop to avoid unnecessary resource consumption.

## ✅ Lab 02 Complete: Recap

* **First Cloud Deployment (europe-west4)**: Running *agents-cli deploy --agent-identity* built and deployed the multi-agent *pulse-concierge* container to **GEAP Agent Runtime** while keeping Gemini Flash and Live inference pinned to the *eu* multi-region endpoint.
* **SPIFFE Agent Identity (*PULSE_AGENT_PRINCIPAL*)**: Your agent runs as a dedicated, system-attested SPIFFE principal and holds Token Creator permissions on *persona-alex-exec* and *persona-elena-care* so BigQuery IAM enforces per-user dataset permissions in the cloud.
* **Three Invocation Surfaces**: You verified the deployed agent over the CLI (in ADK and A2A modes), the Python SDK with persistent server-side **Sessions** and *show_event* tool traces, and the **A2A JSON-RPC protocol**.
* **Catalogued in Agent Registry**: Both *pulse-concierge* (in europe-west4) and Google's managed **BigQuery** and **Knowledge Catalog** MCP servers (in global) are discoverable via Agent Registry.
* **Ready for Fast Local Iteration in Labs 03 to 05**: Subsequent labs reuse the deployed Agent Runtime ID and Agent Identity from *workshop.env* without redeploying the container until Lab 06.

👉 **Next:** Proceed to **[Lab 03: Live MCP Servers (BigQuery + Knowledge Catalog + Custom IAM), Skills & Agent Registry](lab03_mcp_and_skills.ipynb)**.